# 03 — Validation

Read-only view of the validation outputs. Nothing is recomputed here, so the
numbers cannot drift from the code:

- `output/portugal/validation_report.json` — `make validate` (`scripts/run_validation.py`)
- `validation/portugal/calibration_loo.json` — `scripts/fit_calibration.py`

How each source is used (from `stargazing_spots/validate.py`):

| Role | Source | Notes |
|---|---|---|
| Primary | TESS photometers | clear-sky moonless zenith SQM per model pixel (`scripts/derive_tess_clear_sky.py`) |
| Indicative | Published SQM sites | only rows `published_sites_audit.csv` marks `indicative`; not a baseline |
| Model inter-comparison | darkskysites.com grid | VIIRS-based model, not an observation; never validation |
| Descriptive | Official observation points | distance to the nearest pipeline spot |

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("../..").resolve()
REPORT = ROOT / "apps/stargazing_spots/output/portugal/validation_report.json"
CALIB = ROOT / "validation/portugal/calibration_loo.json"

report = json.loads(REPORT.read_text())
calib = json.loads(CALIB.read_text()) if CALIB.exists() else None
METRICS = ["n", "mae", "bias", "rmse", "max_abs_error", "pearson_r", "spearman_rho"]


def metrics_table(entries: dict) -> pd.DataFrame:
    return pd.DataFrame({k: {m: v.get(m) for m in METRICS} for k, v in entries.items()}).T

## 1. TESS clear-sky (primary)

Predicted SQM uses the current `CALIB_FACTOR`, so these are **in-sample**
if the constant was fitted on TESS. See section 2 for held-out error.

In [ ]:
tess = report["primary"].get("tess_clear_sky")
if tess is None:
    print("No TESS result: run scripts/derive_tess_clear_sky.py, then make validate")
else:
    display(metrics_table({"vs KDE mode": tess, "vs median": tess["sensitivity_vs_median"]}))
    sites = pd.DataFrame(tess["sites"])
    sites["error"] = sites["predicted"] - sites["observed_mode"]
    display(sites[["location", "country", "nights", "observed_mode", "predicted", "error"]])
    print("Outside raster:", tess["excluded_outside_raster"])

## 2. Calibration: current constant, in-sample fit, leave-one-out

`CALIB_FACTOR` is the model's only free scale. Leave-one-out fits it on all
TESS pixels but one and predicts the held-out pixel; that error is the
estimate of accuracy at an unseen site.

In [ ]:
if calib is None:
    print("Run scripts/fit_calibration.py")
else:
    table = metrics_table(
        {k: calib[k] for k in ("current", "in_sample_fit", "leave_one_out")}
    )
    table["calib_factor"] = [
        calib["current"]["calib_factor"],
        calib["in_sample_fit"]["calib_factor"],
        f'{calib["leave_one_out"]["calib_factor_min"]}–{calib["leave_one_out"]["calib_factor_max"]}',
    ]
    display(table)

    s = pd.DataFrame(calib["sites"])
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(s["observed_mode"], s["predicted_current"], label="current constant")
    ax.scatter(s["observed_mode"], s["predicted_loo"], marker="x", label="leave-one-out")
    lo, hi = s[["observed_mode", "predicted_current", "predicted_loo"]].stack().agg(["min", "max"])
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel("TESS clear-sky SQM (mag/arcsec²)")
    ax.set_ylabel("Predicted SQM (mag/arcsec²)")
    ax.legend()
    plt.show()

## 3. Published SQM sites (indicative)

Per-site value or location could not be verified; reported separately and
never pooled with TESS.

In [ ]:
pub = report["indicative"].get("published_sqm")
if pub:
    display(metrics_table({"published (indicative)": pub}))
    display(pd.DataFrame(pub["sites"]))
    display(pd.Series(pub["excluded"], name="excluded: reason").to_frame())

## 4. darkskysites.com grid (model inter-comparison, not validation)

Both models are driven by VIIRS, so agreement measures consistency with
another model, not accuracy against the sky.

In [ ]:
grid = report["model_intercomparison"].get("darkskysites_grid")
if grid:
    display(metrics_table({"darkskysites grid": grid}))

## 5. Official observation points (descriptive)

Distance from each official/public observation point to the nearest
pipeline spot.

In [ ]:
op = report["descriptive"].get("official_points_to_nearest_spot_km")
if op:
    print(f'n={op["n"]}  median={op["median"]} km  max={op["max"]} km')
    display(pd.Series(op["per_point"], name="km").to_frame().T)

## Limitations

See `docs/methodology.md` (sections 15–16) and `docs/validation_report.md`.